In [2]:
import pandas as pd
import re
import string
import unicodedata
import emoji

pd.set_option("display.max_colwidth", None)

In [3]:
df = pd.read_csv("EDA_dataset.csv")
df.head()

,reviewId,content,score,at,bank,year
0,46a9b775-12db-4cad-a259-83bdd6ff5b78,bca gaje sinyal bagus tapi merah mulu,1,2025-12-30 19:24:57,BCAMOBILE_REVIEWS,2025
1,fb449431-50de-41a5-9809-df47d670ee2a,apa ini aplikasi bintang 1.. register mobile banking dari pagi Ampe sore bilangnya tunggu indikator hijau... pdahal tidak ada indikator sama sekali Ampe habis pulsa gue 20k,1,2025-12-30 19:23:00,BCAMOBILE_REVIEWS,2025
2,1390039e-cd94-4046-b864-523acc647bc9,akun nya ga bis akembali,1,2025-12-30 17:11:35,BCAMOBILE_REVIEWS,2025
3,5fa82eb0-e111-4a8f-b397-8a6fd37d1709,makin paraaaaaah setelah diupdate .aplikasi nya ya Allah,1,2025-12-30 16:51:09,BCAMOBILE_REVIEWS,2025
4,6881a68b-b85e-4ba0-9d8b-d5a2c21a6ca4,"indikator lampu merah terus , mana sering gitu Mulu lagi, setiap mau dipakek kayak gitu Mulu, sampai berjam2 pada hak sinyal bagus, apk udah diperbarui. kejadian terus berulang² dan sering",1,2025-12-30 16:02:42,BCAMOBILE_REVIEWS,2025


empty review removal

In [4]:
print(f"Before : {len(df):,}")

df = df[
    df["content"]
    .fillna("")
    .str.strip()
    .ne("")
].copy()

print(f"After  : {len(df):,}")

Before : 59,996
After  : 59,996


Noise Review Detection (Inspection Only)

In [5]:
noise_pattern = r"^[A-Za-z\s.,!?;:'\"()\-/\\]+$"

potential_noise = df[
    df["content"]
    .str.fullmatch(noise_pattern, na=False)
]

potential_noise.sample(20, random_state=42)

,reviewId,content,score,at,bank,year
32943,b9743eca-95bf-4aee-aa73-b6b2a888fc75,mengganggu kegiatan saya,1,2025-10-14 06:23:35,LIVIN_MANDIRI_REVIEWS,2025
42175,54051b71-baeb-4e62-a8f0-52125d9fc0eb,"makin aneh ni livin, buka aplikasi minta update, gilirian diupdate, minta update lagi, pusing",1,2025-03-19 13:03:15,LIVIN_MANDIRI_REVIEWS,2025
53171,df442697-cbf6-4be4-8895-c795e935060f,Transfer va lama. payah,1,2025-08-15 18:55:03,WONDR_BNI_REVIEWS,2025
24430,b03eec56-9115-4b48-a937-d833be311b0f,"setelah update brimo gw keluar"" sendiri udah hapus instal lagi tetap SMA keluar sendiri",1,2025-03-19 19:29:03,BRIMO_REVIEWS,2025
58597,66c6b338-2434-4fc5-8de4-28851ee66ba6,"BNI SEMENJAK PINDAH APLIKASI INI, GANGGUAN TERUSA",1,2025-02-13 02:18:55,WONDR_BNI_REVIEWS,2025
49284,eb87e5ed-c09b-4194-a164-2fe150d4692c,biji susah login,3,2025-12-01 13:15:37,WONDR_BNI_REVIEWS,2025
9491,42bf6fed-8afc-42d6-947b-0ff3738a4ab5,"nyebelin banget, harusnya kalau nomor ponsel tidak aktif harusnya suruh verifikasi dulu ganti nomor ponsel, kasih fiturnya! jangan malah langsung hapus akun otomatis sembarangan! bodoh banget si kalian",1,2025-12-22 12:34:11,BRIMO_REVIEWS,2025
29874,c1f577ba-75ef-4785-a8ba-895930e34e90,"Kata CS nya ga support hp realme, oppo , aplikasi apaan ini, saldo ta pindahin ke BCA semua",1,2025-01-01 09:48:01,BRIMO_REVIEWS,2025
26008,11495530-b70d-4e78-b46a-9c387bfa3ca2,"saya mau login username bner ,pasword bener tpi gak bisa ligin tuh gimna apk brimo ini ?",1,2025-03-06 04:14:56,BRIMO_REVIEWS,2025
56881,e7e9172d-1651-4086-8b20-bd7a7f84875f,"kembalikanlah saldo saya, transaksi belum berhasil tetapi saldonya nggak balik. mau itu wondere atau Bni mobile sama aja, kecewa sekali saya. mohon segera perbaiki masalah di aplikasi ini tetapi jika masih belum diperbaiki mungkin banyak orang yang pindah bank yang lebih terjamin sistemnya",1,2025-04-23 18:26:10,WONDR_BNI_REVIEWS,2025


that weird reviews in eda phase isn't here, weird.

case folding

In [6]:
df["case_folding"] = (
    df["content"]
    .astype(str)
    .str.lower()
)

df[
    ["content", "case_folding"]
].head(10)

,content,case_folding
0,bca gaje sinyal bagus tapi merah mulu,bca gaje sinyal bagus tapi merah mulu
1,apa ini aplikasi bintang 1.. register mobile banking dari pagi Ampe sore bilangnya tunggu indikator hijau... pdahal tidak ada indikator sama sekali Ampe habis pulsa gue 20k,apa ini aplikasi bintang 1.. register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau... pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k
2,akun nya ga bis akembali,akun nya ga bis akembali
3,makin paraaaaaah setelah diupdate .aplikasi nya ya Allah,makin paraaaaaah setelah diupdate .aplikasi nya ya allah
4,"indikator lampu merah terus , mana sering gitu Mulu lagi, setiap mau dipakek kayak gitu Mulu, sampai berjam2 pada hak sinyal bagus, apk udah diperbarui. kejadian terus berulang² dan sering","indikator lampu merah terus , mana sering gitu mulu lagi, setiap mau dipakek kayak gitu mulu, sampai berjam2 pada hak sinyal bagus, apk udah diperbarui. kejadian terus berulang² dan sering"
5,"kenapa yaa tiap mau buat m-bca selalu stuck di pembuatan sandi m-bca, pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada, udh coba install uninstall, restart hp ttp ga mau, jaringan wifi sama data juga oke ga ada kendala. ulang² trus smpai brp kali isi pulsa. pdhl dri dulu suka bgt pake ni aplikasi krna simpel, ini ke reset karna ganti no hp. Tolong dongg solusinya","kenapa yaa tiap mau buat m-bca selalu stuck di pembuatan sandi m-bca, pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada, udh coba install uninstall, restart hp ttp ga mau, jaringan wifi sama data juga oke ga ada kendala. ulang² trus smpai brp kali isi pulsa. pdhl dri dulu suka bgt pake ni aplikasi krna simpel, ini ke reset karna ganti no hp. tolong dongg solusinya"
6,indikator merah mulu setelah update,indikator merah mulu setelah update
7,"TERLALU BANYAK ERROR, KIRAIN ADA UPDATE'AN LAGI TERNYATA TIDAK","terlalu banyak error, kirain ada update'an lagi ternyata tidak"
8,"kenapa dari kemarin nggak bisa login, apa sedang gangguan kah","kenapa dari kemarin nggak bisa login, apa sedang gangguan kah"
9,sampah,sampah


checking the data first before continuing 

In [7]:
df[
    df["case_folding"].str.contains(
        r"http|www\.",
        regex=True,
        na=False
    )
][["content"]]

,content


In [8]:
df[
    df["case_folding"].str.contains(
        r"<.*?>",
        regex=True,
        na=False
    )
][["content"]]

,content


In [9]:
df[
    df["case_folding"].str.contains(
        "@",
        regex=False,
        na=False
    )
][["content"]]

,content
2990,Masa kalah sama bank J@g0 . transaksi gagal otomatis langsung refund detik itu juga padahal bank besar.BCA Transaksi Qris ada kndala saldo sudh terpotong. tapi di merchant belum masuk dan Saldo belum balik lagi . telpon Cs jawabannya slalu sama semua
10731,"pelayanan cs nya tidak memuaskan sama sekali kalo ada nasabah mengajukan problem saat menggunakan aplikasi mereka dan proses pengajuan nya juga tidak di tindak lanjuti lagi sama mereka dan sudah 2 hari laporan saya ajukan tidak ada perkembangan nya. begitu juga pelayanan lewat wa mereka low respon banget, kebanyakan boot yang merespon. klo kayak gini terus kedepannya saya masih mempertimbangkan masih mau percaya tidak sama kalian @BRI"
11897,"saya melakukan transaksi pembayaran qris 2 kali percobaan gagal, tapi saldo saya terpotong, sudah hubungi contact BRI, via Wa, @mail hasilnya nihil. kacau ni brimo"
12347,f5.YG SUDAH BAYAR 50K 1.BPK KADIM 200 2.HUMAM 150 3.ADI GENDUT 100 4.HERI 50 5.PANGAT 50 6.AJIS 50 7.ILHAM 50 8 8.FAISAL 50 9.AGENG @ Ono ceritane cah ndugal mlayu Seko pacobaning Urip.List baju 1.wono 19(L) 120 2 Araujoo 27 (M) 120 3 Ageng 12 M 4 piszz 23(S DEWASA) 5 Y. R 14 (m) 6 R. Y 4 (L) 7 ILHAM 22 50 9.AGENG 50 10.DEWAN 50 11.NANOK 50 12.WAWAN 50 136556n6
12847,saya melakukan transfer di aplikasi brimo dengan menggunakan @nama...karna saya fikir yang muncul adalah nama yang biasa bertransaksi dengan kita ternyata RANDOM dan saya ga ngecek kembali nomor rekening tujuan saya alhasil salah transfer dan sampe sekarang uang nya ga balik udah beberapa kali ke kantor cabang tapi katanya orangnya ga bisa dihubungiiii....😭😭😭😭😭😭 9 jt
15248,Kenapa login gagal trs ya @brimo setelah download lagi di beda negara. Boleh tolong saya kak
16451,apk mobil banking paling parah! transfer sesama bri pake @ alias2 Gasut!!! no rekening sudah lengkap tombol lanjutnya masih buram gk bisa di klik
16516,"dapat notif : We detected something blocking the application screen. To make sure your data is safe, please close all popups and other overlays. If nothing is obstructing the application screen, try to restart the application and perform a malware scan. sudah sampai install ulang, reeboot, bahkan juga ke CS BRI langsung tetap tidak ada solusi, tutorial youtube, dll dkk, tetep tidak bisa buat transaksi dan muncul notif itu😭😭 #BRImo @BRImo"
17848,"saya mau isi saldo shoopay kok susah sekali ya, kalau mau isi saldo 2 juta aja ribet amat, hari ini trfer 1 juta pkek @wallet bisa, ntar kelang brpa jam gk bisa lagi. trz aku tfer 5rtus ribu bis, stlh itu gk bisa lagi, trz kirim lagi 3ratus bisa, trs sisanya mau trfer 2 ratus gk bisa lagi, jadi aku tf 100rbu bisa, mau tf lgi gk bisa lagi. adu pusing ya susah bner mau trfer ke shoopay aja ribet gitu. sebelumnya gk ada masalah mau tf brpa aja ke shoopay. sekarang ribet pusing gk tau salahnya dimna"
19794,"luar bi@sa semenjak menggunakan BRimo mudah melakukan transaksi, mantap"


unicoe normalization  
normalizing weird fonts

In [10]:
def normalize_unicode(text):
    if not isinstance(text, str):
        return text
    return unicodedata.normalize("NFKC", text)

df["unicode_normalized"] = df["case_folding"].apply(normalize_unicode)

unicode_changed = df[
    df["case_folding"] != df["unicode_normalized"]
]

print(f"Total changed rows: {len(unicode_changed):,}")

unicode_changed[
    [
        "case_folding",
        "unicode_normalized"
    ]
].head(10)

Total changed rows: 1,325


,case_folding,unicode_normalized
4,"indikator lampu merah terus , mana sering gitu mulu lagi, setiap mau dipakek kayak gitu mulu, sampai berjam2 pada hak sinyal bagus, apk udah diperbarui. kejadian terus berulang² dan sering","indikator lampu merah terus , mana sering gitu mulu lagi, setiap mau dipakek kayak gitu mulu, sampai berjam2 pada hak sinyal bagus, apk udah diperbarui. kejadian terus berulang2 dan sering"
5,"kenapa yaa tiap mau buat m-bca selalu stuck di pembuatan sandi m-bca, pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada, udh coba install uninstall, restart hp ttp ga mau, jaringan wifi sama data juga oke ga ada kendala. ulang² trus smpai brp kali isi pulsa. pdhl dri dulu suka bgt pake ni aplikasi krna simpel, ini ke reset karna ganti no hp. tolong dongg solusinya","kenapa yaa tiap mau buat m-bca selalu stuck di pembuatan sandi m-bca, pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada, udh coba install uninstall, restart hp ttp ga mau, jaringan wifi sama data juga oke ga ada kendala. ulang2 trus smpai brp kali isi pulsa. pdhl dri dulu suka bgt pake ni aplikasi krna simpel, ini ke reset karna ganti no hp. tolong dongg solusinya"
12,semenjak di update jadi lama bgt kalau mau masuk mbanking nya mau bayar² pake qris juga jadi males,semenjak di update jadi lama bgt kalau mau masuk mbanking nya mau bayar2 pake qris juga jadi males
88,"saldo saya tiba² ilang tanpa ada mutasi padahal kemarin baru kepotong bunga 10rb perbulan, apk aneh","saldo saya tiba2 ilang tanpa ada mutasi padahal kemarin baru kepotong bunga 10rb perbulan, apk aneh"
101,"kenapa lemot sekali mau buka apk nya..pdhal jaringan bagus,tpi jaringan terputus terus kira² knpa ya apa lagi eror","kenapa lemot sekali mau buka apk nya..pdhal jaringan bagus,tpi jaringan terputus terus kira2 knpa ya apa lagi eror"
116,tau² oneklik dinonaktifkan aja ngga jelas apa alasannya.masa iya hanya sebulan ngga transaksi langsung diblokir..ngeselin,tau2 oneklik dinonaktifkan aja ngga jelas apa alasannya.masa iya hanya sebulan ngga transaksi langsung diblokir..ngeselin
289,"setelah update sering muncul indikator merah, sebelumnya lancar² saja dan qris loading sangat lama, bahkan sama sekali ga terbuka. mohon diperbaiki.","setelah update sering muncul indikator merah, sebelumnya lancar2 saja dan qris loading sangat lama, bahkan sama sekali ga terbuka. mohon diperbaiki."
325,indikator warna yg mengganggu sinyal ga cocok dikit gabisa dibuat apa² apknya,indikator warna yg mengganggu sinyal ga cocok dikit gabisa dibuat apa2 apknya
391,"kebiasaan kali sering gangguan pas malam minggu,udah gitu lampu indikator merahnya lama lgi anj,sebelum update aja dah mntp,gk usah di-update klw malah nambah masalah,mbankking dipakek cuma buat byr,ngirim scan brcode doang..gk usah diperberat atau nambah² inj itu jnck","kebiasaan kali sering gangguan pas malam minggu,udah gitu lampu indikator merahnya lama lgi anj,sebelum update aja dah mntp,gk usah di-update klw malah nambah masalah,mbankking dipakek cuma buat byr,ngirim scan brcode doang..gk usah diperberat atau nambah2 inj itu jnck"
416,"semenjak update versi terbaru jadi sangat² berat, mau cek saldo aja lama dan melakukan transaksi juga lama.. mohon segera diperbaiki","semenjak update versi terbaru jadi sangat2 berat, mau cek saldo aja lama dan melakukan transaksi juga lama.. mohon segera diperbaiki"


remove emoji

In [11]:
df["remove_emoji"] = (
    df["unicode_normalized"]
    .apply(lambda text: emoji.replace_emoji(text, replace=""))
)

In [12]:
emoji_changed = df[
    df["unicode_normalized"] != df["remove_emoji"]
][["unicode_normalized", "remove_emoji"]]

print(f"Rows affected: {len(emoji_changed)}")

emoji_changed.head(10)

Rows affected: 3199


,unicode_normalized,remove_emoji
38,"yg terupdate tiap login kenapa mesti 2x siih? login pertama setelah input kode akses, pasti auto belum masuk dan harus ulangi ketik lagi barulah bisa masuk. selalu kaya gitu lohh di versi terbaru ini. padahal versi sebelumnya gak pernah gini. normal aja sekali login langsung masuk. mohon penjelasannya min 🙏","yg terupdate tiap login kenapa mesti 2x siih? login pertama setelah input kode akses, pasti auto belum masuk dan harus ulangi ketik lagi barulah bisa masuk. selalu kaya gitu lohh di versi terbaru ini. padahal versi sebelumnya gak pernah gini. normal aja sekali login langsung masuk. mohon penjelasannya min"
59,habia update malah qris nya lola 🤦,habia update malah qris nya lola
69,mohon bertanya min saya pindah kota ke jawa ko mobile bca sya gk bisa di buka suruh masuk nomor kartu atm tidak ada konfirmasi lewat nomor hp min susah mau isi saldo mlhn eror begini min🙇,mohon bertanya min saya pindah kota ke jawa ko mobile bca sya gk bisa di buka suruh masuk nomor kartu atm tidak ada konfirmasi lewat nomor hp min susah mau isi saldo mlhn eror begini min
71,"mau login ke bca mobile susah, sampai harus ber puluh x tetep aj gak bisa padahal pulsa banyak. gak ad sulusi solusi bantuan g7🤨","mau login ke bca mobile susah, sampai harus ber puluh x tetep aj gak bisa padahal pulsa banyak. gak ad sulusi solusi bantuan g7"
98,"registrasi susah. gagal trs, jelek banget bca mobile skrg 👎👎","registrasi susah. gagal trs, jelek banget bca mobile skrg"
122,"kenapa di hp saya tidak bisa di buka , alasan jaringan bermasalah padhal wifi saya bagus mohon di perbaiki jika sudah saya akan ganti ke ⭐5","kenapa di hp saya tidak bisa di buka , alasan jaringan bermasalah padhal wifi saya bagus mohon di perbaiki jika sudah saya akan ganti ke 5"
132,aku gak bisa masuk akun lama aku 😭😖🙏,aku gak bisa masuk akun lama aku
174,bca mobile terblokir tanpa sebab musabab ada kesalahan pin atau apalah. akhirnya ngrepotin nasabah harus antri panjang ke cs💩💩,bca mobile terblokir tanpa sebab musabab ada kesalahan pin atau apalah. akhirnya ngrepotin nasabah harus antri panjang ke cs
182,"keluar"" mlu dh verifikasi mlu sms😠ud verifikasi wajah masi gagal mlu pulsa habis buat verifikasi bca doang","keluar"" mlu dh verifikasi mlu smsud verifikasi wajah masi gagal mlu pulsa habis buat verifikasi bca doang"
197,"mohon maaf ya bintang satu dulu, notifikasi transfer masuk tidak muncul☺","mohon maaf ya bintang satu dulu, notifikasi transfer masuk tidak muncul"


remove "@"

In [13]:
df["remove_at"] = (
    df["remove_emoji"]
    .str.replace("@", "", regex=False)
)

In [14]:
at_changed = df[
    df["remove_emoji"] != df["remove_at"]
][["remove_emoji", "remove_at"]]

print(f"Rows affected: {len(at_changed):,}")

at_changed.head(10)

Rows affected: 31


,remove_emoji,remove_at
2990,masa kalah sama bank j@g0 . transaksi gagal otomatis langsung refund detik itu juga padahal bank besar.bca transaksi qris ada kndala saldo sudh terpotong. tapi di merchant belum masuk dan saldo belum balik lagi . telpon cs jawabannya slalu sama semua,masa kalah sama bank jg0 . transaksi gagal otomatis langsung refund detik itu juga padahal bank besar.bca transaksi qris ada kndala saldo sudh terpotong. tapi di merchant belum masuk dan saldo belum balik lagi . telpon cs jawabannya slalu sama semua
10731,"pelayanan cs nya tidak memuaskan sama sekali kalo ada nasabah mengajukan problem saat menggunakan aplikasi mereka dan proses pengajuan nya juga tidak di tindak lanjuti lagi sama mereka dan sudah 2 hari laporan saya ajukan tidak ada perkembangan nya. begitu juga pelayanan lewat wa mereka low respon banget, kebanyakan boot yang merespon. klo kayak gini terus kedepannya saya masih mempertimbangkan masih mau percaya tidak sama kalian @bri","pelayanan cs nya tidak memuaskan sama sekali kalo ada nasabah mengajukan problem saat menggunakan aplikasi mereka dan proses pengajuan nya juga tidak di tindak lanjuti lagi sama mereka dan sudah 2 hari laporan saya ajukan tidak ada perkembangan nya. begitu juga pelayanan lewat wa mereka low respon banget, kebanyakan boot yang merespon. klo kayak gini terus kedepannya saya masih mempertimbangkan masih mau percaya tidak sama kalian bri"
11897,"saya melakukan transaksi pembayaran qris 2 kali percobaan gagal, tapi saldo saya terpotong, sudah hubungi contact bri, via wa, @mail hasilnya nihil. kacau ni brimo","saya melakukan transaksi pembayaran qris 2 kali percobaan gagal, tapi saldo saya terpotong, sudah hubungi contact bri, via wa, mail hasilnya nihil. kacau ni brimo"
12347,f5.yg sudah bayar 50k 1.bpk kadim 200 2.humam 150 3.adi gendut 100 4.heri 50 5.pangat 50 6.ajis 50 7.ilham 50 8 8.faisal 50 9.ageng @ ono ceritane cah ndugal mlayu seko pacobaning urip.list baju 1.wono 19(l) 120 2 araujoo 27 (m) 120 3 ageng 12 m 4 piszz 23(s dewasa) 5 y. r 14 (m) 6 r. y 4 (l) 7 ilham 22 50 9.ageng 50 10.dewan 50 11.nanok 50 12.wawan 50 136556n6,f5.yg sudah bayar 50k 1.bpk kadim 200 2.humam 150 3.adi gendut 100 4.heri 50 5.pangat 50 6.ajis 50 7.ilham 50 8 8.faisal 50 9.ageng ono ceritane cah ndugal mlayu seko pacobaning urip.list baju 1.wono 19(l) 120 2 araujoo 27 (m) 120 3 ageng 12 m 4 piszz 23(s dewasa) 5 y. r 14 (m) 6 r. y 4 (l) 7 ilham 22 50 9.ageng 50 10.dewan 50 11.nanok 50 12.wawan 50 136556n6
12847,saya melakukan transfer di aplikasi brimo dengan menggunakan @nama...karna saya fikir yang muncul adalah nama yang biasa bertransaksi dengan kita ternyata random dan saya ga ngecek kembali nomor rekening tujuan saya alhasil salah transfer dan sampe sekarang uang nya ga balik udah beberapa kali ke kantor cabang tapi katanya orangnya ga bisa dihubungiiii.... 9 jt,saya melakukan transfer di aplikasi brimo dengan menggunakan nama...karna saya fikir yang muncul adalah nama yang biasa bertransaksi dengan kita ternyata random dan saya ga ngecek kembali nomor rekening tujuan saya alhasil salah transfer dan sampe sekarang uang nya ga balik udah beberapa kali ke kantor cabang tapi katanya orangnya ga bisa dihubungiiii.... 9 jt
15248,kenapa login gagal trs ya @brimo setelah download lagi di beda negara. boleh tolong saya kak,kenapa login gagal trs ya brimo setelah download lagi di beda negara. boleh tolong saya kak
16451,apk mobil banking paling parah! transfer sesama bri pake @ alias2 gasut!!! no rekening sudah lengkap tombol lanjutnya masih buram gk bisa di klik,apk mobil banking paling parah! transfer sesama bri pake alias2 gasut!!! no rekening sudah lengkap tombol lanjutnya masih buram gk bisa di klik
16516,"dapat notif : we detected something blocking the application screen. to make sure your data is safe, please close all popups and other overlays. if nothing is obstructing the application screen, try to restart the application and perform a malware scan. sudah sampai install ulang, reebo

remove punctuation

In [15]:
df["remove_punctuation"] = (
    df["remove_at"]
    .str.replace(r"[^\w\s]", " ", regex=True)
)

In [16]:
punctuation_changed = df[
    df["remove_at"] != df["remove_punctuation"]
][["remove_at", "remove_punctuation"]]

print(f"Rows affected: {len(punctuation_changed):,}")

punctuation_changed.head(10)

Rows affected: 37,848


,remove_at,remove_punctuation
1,apa ini aplikasi bintang 1.. register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau... pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k,apa ini aplikasi bintang 1 register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k
3,makin paraaaaaah setelah diupdate .aplikasi nya ya allah,makin paraaaaaah setelah diupdate aplikasi nya ya allah
4,"indikator lampu merah terus , mana sering gitu mulu lagi, setiap mau dipakek kayak gitu mulu, sampai berjam2 pada hak sinyal bagus, apk udah diperbarui. kejadian terus berulang2 dan sering",indikator lampu merah terus mana sering gitu mulu lagi setiap mau dipakek kayak gitu mulu sampai berjam2 pada hak sinyal bagus apk udah diperbarui kejadian terus berulang2 dan sering
5,"kenapa yaa tiap mau buat m-bca selalu stuck di pembuatan sandi m-bca, pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada, udh coba install uninstall, restart hp ttp ga mau, jaringan wifi sama data juga oke ga ada kendala. ulang2 trus smpai brp kali isi pulsa. pdhl dri dulu suka bgt pake ni aplikasi krna simpel, ini ke reset karna ganti no hp. tolong dongg solusinya",kenapa yaa tiap mau buat m bca selalu stuck di pembuatan sandi m bca pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada udh coba install uninstall restart hp ttp ga mau jaringan wifi sama data juga oke ga ada kendala ulang2 trus smpai brp kali isi pulsa pdhl dri dulu suka bgt pake ni aplikasi krna simpel ini ke reset karna ganti no hp tolong dongg solusinya
7,"terlalu banyak error, kirain ada update'an lagi ternyata tidak",terlalu banyak error kirain ada update an lagi ternyata tidak
8,"kenapa dari kemarin nggak bisa login, apa sedang gangguan kah",kenapa dari kemarin nggak bisa login apa sedang gangguan kah
10,"payah ,jauh sama livin mandiri. saat kirim sms kode, tapi gak masuk masuk hanya muter aja",payah jauh sama livin mandiri saat kirim sms kode tapi gak masuk masuk hanya muter aja
11,qris bca sekarang lemot bgt..,qris bca sekarang lemot bgt
14,mbca skrg tdk kayak dulu lancar.. sekarang sering bug lemot,mbca skrg tdk kayak dulu lancar sekarang sering bug lemot
15,"verifikasi e-ktp dan wajah selalu gagal, padahal data udah benar",verifikasi e ktp dan wajah selalu gagal padahal data udah benar


remove extra whitespace

In [17]:
df["cleaned"] = (
    df["remove_punctuation"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [18]:
space_changed = df[
    df["remove_punctuation"] != df["cleaned"]
][["remove_punctuation", "cleaned"]]

print(f"Rows affected: {len(space_changed):,}")

space_changed.head(10)

Rows affected: 35,676


,remove_punctuation,cleaned
1,apa ini aplikasi bintang 1 register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k,apa ini aplikasi bintang 1 register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k
3,makin paraaaaaah setelah diupdate aplikasi nya ya allah,makin paraaaaaah setelah diupdate aplikasi nya ya allah
4,indikator lampu merah terus mana sering gitu mulu lagi setiap mau dipakek kayak gitu mulu sampai berjam2 pada hak sinyal bagus apk udah diperbarui kejadian terus berulang2 dan sering,indikator lampu merah terus mana sering gitu mulu lagi setiap mau dipakek kayak gitu mulu sampai berjam2 pada hak sinyal bagus apk udah diperbarui kejadian terus berulang2 dan sering
5,kenapa yaa tiap mau buat m bca selalu stuck di pembuatan sandi m bca pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada udh coba install uninstall restart hp ttp ga mau jaringan wifi sama data juga oke ga ada kendala ulang2 trus smpai brp kali isi pulsa pdhl dri dulu suka bgt pake ni aplikasi krna simpel ini ke reset karna ganti no hp tolong dongg solusinya,kenapa yaa tiap mau buat m bca selalu stuck di pembuatan sandi m bca pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada udh coba install uninstall restart hp ttp ga mau jaringan wifi sama data juga oke ga ada kendala ulang2 trus smpai brp kali isi pulsa pdhl dri dulu suka bgt pake ni aplikasi krna simpel ini ke reset karna ganti no hp tolong dongg solusinya
7,terlalu banyak error kirain ada update an lagi ternyata tidak,terlalu banyak error kirain ada update an lagi ternyata tidak
8,kenapa dari kemarin nggak bisa login apa sedang gangguan kah,kenapa dari kemarin nggak bisa login apa sedang gangguan kah
10,payah jauh sama livin mandiri saat kirim sms kode tapi gak masuk masuk hanya muter aja,payah jauh sama livin mandiri saat kirim sms kode tapi gak masuk masuk hanya muter aja
11,qris bca sekarang lemot bgt,qris bca sekarang lemot bgt
14,mbca skrg tdk kayak dulu lancar sekarang sering bug lemot,mbca skrg tdk kayak dulu lancar sekarang sering bug lemot
15,verifikasi e ktp dan wajah selalu gagal padahal data udah benar,verifikasi e ktp dan wajah selalu gagal padahal data udah benar


remove empty reviews

In [19]:
before = len(df)

df = df[
    df["cleaned"]
    .str.strip()
    .ne("")
].copy()

after = len(df)

print(f"Before : {before:,}")
print(f"After  : {after:,}")
print(f"Removed: {before-after:,}")

Before : 59,996
After  : 59,952
Removed: 44


comparison

In [20]:
comparison = df.copy()

comparison["changes"] = (
    (comparison["content"] != comparison["case_folding"]).astype(int) +
    (comparison["case_folding"] != comparison["unicode_normalized"]).astype(int) +
    (comparison["unicode_normalized"] != comparison["remove_emoji"]).astype(int) +
    (comparison["remove_emoji"] != comparison["remove_at"]).astype(int) +
    (comparison["remove_at"] != comparison["remove_punctuation"]).astype(int) +
    (comparison["remove_punctuation"] != comparison["cleaned"]).astype(int)
)

comparison = comparison.sort_values(
    by="changes",
    ascending=False
)

comparison[
    [
        "changes",
        "content",
        "case_folding",
        "unicode_normalized",
        "remove_emoji",
        "remove_at",
        "remove_punctuation",
        "cleaned"
    ]
].head(10)

,changes,content,case_folding,unicode_normalized,remove_emoji,remove_at,remove_punctuation,cleaned
28513,5,"Dari tahun 2023 sampe skrg daftar brimo gk bisa²,padahal rekening ada,alasan apk jaringanlah eror lah,pdhl wifi bagus,jaringan pun bagus,pelayanan mcm apa ini,bukannya mempermudah 😑😑","dari tahun 2023 sampe skrg daftar brimo gk bisa²,padahal rekening ada,alasan apk jaringanlah eror lah,pdhl wifi bagus,jaringan pun bagus,pelayanan mcm apa ini,bukannya mempermudah 😑😑","dari tahun 2023 sampe skrg daftar brimo gk bisa2,padahal rekening ada,alasan apk jaringanlah eror lah,pdhl wifi bagus,jaringan pun bagus,pelayanan mcm apa ini,bukannya mempermudah 😑😑","dari tahun 2023 sampe skrg daftar brimo gk bisa2,padahal rekening ada,alasan apk jaringanlah eror lah,pdhl wifi bagus,jaringan pun bagus,pelayanan mcm apa ini,bukannya mempermudah","dari tahun 2023 sampe skrg daftar brimo gk bisa2,padahal rekening ada,alasan apk jaringanlah eror lah,pdhl wifi bagus,jaringan pun bagus,pelayanan mcm apa ini,bukannya mempermudah",dari tahun 2023 sampe skrg daftar brimo gk bisa2 padahal rekening ada alasan apk jaringanlah eror lah pdhl wifi bagus jaringan pun bagus pelayanan mcm apa ini bukannya mempermudah,dari tahun 2023 sampe skrg daftar brimo gk bisa2 padahal rekening ada alasan apk jaringanlah eror lah pdhl wifi bagus jaringan pun bagus pelayanan mcm apa ini bukannya mempermudah
52524,5,"mentang² bnyk yng pake potongan dlem nya gede bgt, gmn kl yng lg nganggur blm smpt krja lg, blm smpt lg pke atmn ny, aku ksi 3 bintng dlu, msa mau di ptong trs, nti² nymoe smpe 0% lgi smpe k blok, bbro nge cek/isi min gmna kl uang cm buat kbtuhn, gmungkin ingt isi sldo wlpun cm 10rb/gdeny 30rb, tlong lahk min bntu kringanan dkit, kita enk pke atm ni BNI, bgus jg sbnrny, msa mkin ksni mkin ksna si🙏🏼","mentang² bnyk yng pake potongan dlem nya gede bgt, gmn kl yng lg nganggur blm smpt krja lg, blm smpt lg pke atmn ny, aku ksi 3 bintng dlu, msa mau di ptong trs, nti² nymoe smpe 0% lgi smpe k blok, bbro nge cek/isi min gmna kl uang cm buat kbtuhn, gmungkin ingt isi sldo wlpun cm 10rb/gdeny 30rb, tlong lahk min bntu kringanan dkit, kita enk pke atm ni bni, bgus jg sbnrny, msa mkin ksni mkin ksna si🙏🏼","mentang2 bnyk yng pake potongan dlem nya gede bgt, gmn kl yng lg nganggur blm smpt krja lg, blm smpt lg pke atmn ny, aku ksi 3 bintng dlu, msa mau di ptong trs, nti2 nymoe smpe 0% lgi smpe k blok, bbro nge cek/isi min gmna kl uang cm buat kbtuhn, gmungkin ingt isi sldo wlpun cm 10rb/gdeny 30rb, tlong lahk min bntu kringanan dkit, kita enk pke atm ni bni, bgus jg sbnrny, msa mkin ksni mkin ksna si🙏🏼","mentang2 bnyk yng pake potongan dlem nya gede bgt, gmn kl yng lg nganggur blm smpt krja lg, blm smpt lg pke atmn ny, aku ksi 3 bintng dlu, msa mau di ptong trs, nti2 nymoe smpe 0% lgi smpe k blok, bbro nge cek/isi min gmna kl uang cm buat kbtuhn, gmungkin ingt isi sldo wlpun cm 10rb/gdeny 30rb, tlong lahk min bntu kringanan dkit, kita enk pke atm ni bni, bgus jg sbnrny, msa mkin ksni mkin ksna si","mentang2 bnyk yng pake potongan dlem nya gede bgt, gmn kl yng lg nganggur blm smpt krja lg, blm smpt lg pke atmn ny, aku ksi 3 bintng dlu, msa mau di ptong trs, nti2 nymoe smpe 0% lgi smpe k blok, bbro nge cek/isi min gmna kl uang cm buat kbtuhn, gmungkin ingt isi sldo wlpun cm 10rb/gdeny 30rb, tlong lahk min bntu kringanan dkit, kita enk pke atm ni bni, bgus jg sbnrny, msa mkin ksni mkin ksna si",mentang2 bnyk yng pake potongan dlem nya gede bgt gmn kl yng lg nganggur blm smpt krja lg blm smpt lg pke atmn ny aku ksi 3 bintng dlu msa mau di ptong trs nti2 nymoe smpe 0 lgi smpe k blok bbro nge cek isi min gmna kl uang cm buat kbtuhn gmungkin ingt isi sldo wlpun cm 10rb gdeny 30rb tlong lahk min bntu kringanan dkit kita enk pke atm ni bni bgus jg sbnrny msa mkin ksni mkin ksna si,mentang2 bnyk yng pake potongan dlem nya gede bgt gmn kl yng lg nganggur blm smpt krja lg blm smpt lg pke atmn ny aku ksi 3 bintng dlu msa mau di ptong trs nti2 nymoe smpe 0 lgi smpe k blok bbro nge cek

slang words normalization

In [22]:
slang_dict = pd.read_csv("colloquial-indonesian-lexicon.csv")

slang_dict = slang_dict[["slang", "formal"]]

print(len(slang_dict))
slang_dict.head(20)

15006


,slang,formal
0,woww,wow
1,aminn,amin
2,met,selamat
3,netaas,menetas
4,keberpa,keberapa
5,eeeehhhh,eh
6,kata2nyaaa,kata-katanya
7,hallo,halo
8,kaka,kakak
9,ka,kak


In [23]:
slang_dict = dict(
    zip(
        slang_dict["slang"],
        slang_dict["formal"]
    )
)


In [24]:
print(slang_dict["blh"])

boleh


In [25]:
protected_words = {
    # Banks
    "bca",
    "bni",
    "bri",
    "mandiri",

    # Applications
    "brimo",
    "livin",
    "wondr",
    "blu",

    # Banking terms
    "qris",
    "qr",
    "atm",
    "otp",
    "pin",
    "rekening",
    "mbanking",
    "mbanking",
    "m-banking",
    "internet",
    "mobile",
    "token",
    "mtoken",

    # Company names
    "dana",
    "ovo",
    "gopay",
    "linkaja",
    "flip",

    # Payment systems
    "visa",
    "mastercard"
}

In [26]:
# from collections import Counter

# all_words = []

# for text in df["cleaned"]:
#     all_words.extend(text.split())

# word_freq = Counter(all_words)

# vocab_df = (
#     pd.DataFrame(
#         word_freq.items(),
#         columns=["word", "frequency"]
#     )
#     .sort_values(
#         by="frequency",
#         ascending=False
#     )
#         .reset_index(drop=True)
# )

In [27]:
# vocab_df["in_dictionary"] = vocab_df["word"].isin(slang_dict)

# vocab_df["formal"] = vocab_df["word"].map(slang_dict)

# vocab_df.head()

In [28]:
# def classify(row):
#     if row["in_dictionary"]:
#         return "Dictionary"

#     if row["frequency"] >= 20:
#         return "Review"

#     return "Rare"

# vocab_df["status"] = vocab_df.apply(classify, axis=1)

In [29]:
# vocab_df = vocab_df.sort_values(
#     by=[
#         "status",
#         "frequency"
#     ],
#     ascending=[True, False]
# )

In [30]:
# vocab_df["approved"] = ""
# vocab_df["notes"] = ""

# vocab_df.to_csv(
#     "vocabulary_review.csv",
#     index=False,
#     encoding="utf-8-sig"
# )


In [31]:
def normalize_text(text):
    words = text.split()

    normalized_words = []
    changes = 0

    for word in words:

        # Never normalize protected words
        if word in protected_words:
            normalized_words.append(word)
            continue

        new_word = slang_dict.get(word, word)

        if new_word != word:
            changes += 1

        normalized_words.append(new_word)

    return " ".join(normalized_words), changes

In [32]:
result = df["cleaned"].apply(normalize_text)

df["normalized"] = result.str[0]
df["normalization_changes"] = result.str[1]

print(f"Reviews normalized : {(df['normalization_changes']>0).sum():,}")

print()

print(df["normalization_changes"].describe())

Reviews normalized : 39,903

count    59952.000000
mean         2.082499
std          2.766077
min          0.000000
25%          0.000000
50%          1.000000
75%          3.000000
max         40.000000
Name: normalization_changes, dtype: float64


In [33]:
changed_reviews = df[
    df["normalization_changes"] > 0
][[
    "cleaned",
    "normalized",
    "normalization_changes"
]]

changed_reviews.head(20)

,cleaned,normalized,normalization_changes
0,bca gaje sinyal bagus tapi merah mulu,bca enggak jelas sinyal bagus tapi merah mulu,1
1,apa ini aplikasi bintang 1 register mobile banking dari pagi ampe sore bilangnya tunggu indikator hijau pdahal tidak ada indikator sama sekali ampe habis pulsa gue 20k,apa ini aplikasi bintang 1 register mobile banking dari pagi sampai sore bilangnya tunggu indikator hijau padahal tidak ada indikator sama sekali sampai habis pulsa gue 20k,3
2,akun nya ga bis akembali,akun nya enggak bis akembali,1
4,indikator lampu merah terus mana sering gitu mulu lagi setiap mau dipakek kayak gitu mulu sampai berjam2 pada hak sinyal bagus apk udah diperbarui kejadian terus berulang2 dan sering,indikator lampu merah terus mana sering begitu mulu lagi setiap mau dipakek kayak begitu mulu sampai berjam2 pada hak sinyal bagus apk sudah diperbarui kejadian terus berulang2 dan sering,3
5,kenapa yaa tiap mau buat m bca selalu stuck di pembuatan sandi m bca pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya ga ada udh coba install uninstall restart hp ttp ga mau jaringan wifi sama data juga oke ga ada kendala ulang2 trus smpai brp kali isi pulsa pdhl dri dulu suka bgt pake ni aplikasi krna simpel ini ke reset karna ganti no hp tolong dongg solusinya,kenapa ya tiap mau buat sama bca selalu stuck di pembuatan sandi sama bca pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya enggak ada sudah coba install uninstall restart hp tetap enggak mau jaringan wifi sama data juga oke enggak ada kendala ulang2 terus sampai berapa kali isi pulsa padahal dari dulu suka banget pakai nih aplikasi karena simpel ini ke reset karena ganti no hp tolong dong solusinya,19
7,terlalu banyak error kirain ada update an lagi ternyata tidak,terlalu banyak error mengira ada update an lagi ternyata tidak,1
8,kenapa dari kemarin nggak bisa login apa sedang gangguan kah,kenapa dari kemarin enggak bisa login apa sedang gangguan kah,1
10,payah jauh sama livin mandiri saat kirim sms kode tapi gak masuk masuk hanya muter aja,payah jauh sama livin mandiri saat kirim sms kode tapi enggak masuk masuk hanya muter saja,2
11,qris bca sekarang lemot bgt,qris bca sekarang lemot banget,1
12,semenjak di update jadi lama bgt kalau mau masuk mbanking nya mau bayar2 pake qris juga jadi males,semenjak di update jadi lama banget kalau mau masuk mbanking nya mau bayar2 pakai qris juga jadi malas,3


Reduce three or more consecutive identical characters  
example = "paraaahhhhhh" , "jeeelleeekkkk"

In [34]:
def normalize_repeated_chars(text):
    # Reduce 3 or more repeated characters to a single character
    return re.sub(r"(.)\1{2,}", r"\1", text)

def remove_repeated_words(text):
    words = text.split()
    cleaned_words = []

    for word in words:
        if not cleaned_words or word != cleaned_words[-1]:
            cleaned_words.append(word)

    return " ".join(cleaned_words)

def remove_single_character_noise(text):
    return " ".join(
        word for word in text.split()
        if len(word) > 1
    )

df["final_text"] = (
    df["normalized"]
    .apply(normalize_repeated_chars)
    .apply(remove_repeated_words)
    .apply(remove_single_character_noise)
)

In [35]:
df["final_text"] = (
    df["normalized"]
    .apply(normalize_repeated_chars)
    .apply(remove_repeated_words)
    .apply(remove_single_character_noise)
)

In [36]:
df["before_repeat"] = df["normalized"]

def normalize_repeated_chars(text):
    return re.sub(r"(.)\1{2,}", r"\1", text)

df["after_repeat"] = df["normalized"].apply(normalize_repeated_chars)

show = df[
    df["before_repeat"] != df["after_repeat"]
][["before_repeat", "after_repeat"]]

print(f"Jumlah review yang berubah: {len(show):,}")
show.head(20)

Jumlah review yang berubah: 3,061


,before_repeat,after_repeat
3,makin paraaaaaah setelah diupdate aplikasi nya ya allah,makin parah setelah diupdate aplikasi nya ya allah
48,apk baru diupdate malah enggak bisa dibuka kan aneh apk ngennn,apk baru diupdate malah enggak bisa dibuka kan aneh apk ngen
85,kinerja bank nya bagus aplilkasi nya lemot kalau buka qris diperbaiki yang komplain sudah banyak kok masih bilang enggak ada kendala haduhhhh mau jadi bank bumn enggak mau terima kritik 24 12 sudah berapa minggu enggak ada perbaikan qris mu itu mu pecat saja,kinerja bank nya bagus aplilkasi nya lemot kalau buka qris diperbaiki yang komplain sudah banyak kok masih bilang enggak ada kendala haduh mau jadi bank bumn enggak mau terima kritik 24 12 sudah berapa minggu enggak ada perbaikan qris mu itu mu pecat saja
86,jangan terlalu percaya dengan aplikasi saat urgen lampu indikatornya merah terussss kan goblog,jangan terlalu percaya dengan aplikasi saat urgen lampu indikatornya merah terus kan goblog
145,ini bagaimana sih bca tolong dong yang jelas masa potongan bulanan potongnnya enggak jls benar bulan kemarin saldo saya di sedot 25000 sekarang malah 24000 tapi di prosedurnya potongan setiap bulan hanya 15000 bagaimana ini bca aneh,ini bagaimana sih bca tolong dong yang jelas masa potongan bulanan potongnnya enggak jls benar bulan kemarin saldo saya di sedot 250 sekarang malah 240 tapi di prosedurnya potongan setiap bulan hanya 150 bagaimana ini bca aneh
146,ih paling kesel sama bca indikator merah tusss lama lagi ayok di perbaiki padahal sinyal bagus kebiasaan setiap mau pembayaran merah teusss lamaaa lagi,ih paling kesel sama bca indikator merah tus lama lagi ayok di perbaiki padahal sinyal bagus kebiasaan setiap mau pembayaran merah teus lama lagi
152,bca sebagai bank no 1 di indonesia ternyata tidak bisa memberi penjelasan kemana hilangnya uang saya yang tidak tercantum dalam mutasi memang nominalnya hanya 30 000 tapi itu jelas membuktikan keteledoran pihak bca dalam menjaga uang nasabahnya bagaimana jika 30 000 dikalikan ribuan orang lain yang enggak sadar uangnya terpotong enggak jelas tolong bca jelaskan kemana uang saya itu,bca sebagai bank no 1 di indonesia ternyata tidak bisa memberi penjelasan kemana hilangnya uang saya yang tidak tercantum dalam mutasi memang nominalnya hanya 30 0 tapi itu jelas membuktikan keteledoran pihak bca dalam menjaga uang nasabahnya bagaimana jika 30 0 dikalikan ribuan orang lain yang enggak sadar uangnya terpotong enggak jelas tolong bca jelaskan kemana uang saya itu
170,ini kenapa ya setiap mau masuk apk selalu ada tulisan 205 transaksi tidak dapat diproses coba lagi nanti kenapaaaaa saya mau cek saldo mau transfer padahal sinyal bagus sudah beberapa kali restart hp tolong diperbaiki secepatnya,ini kenapa ya setiap mau masuk apk selalu ada tulisan 205 transaksi tidak dapat diproses coba lagi nanti kenapa saya mau cek saldo mau transfer padahal sinyal bagus sudah beberapa kali restart hp tolong diperbaiki secepatnya
201,wahaii pemuda bikin aplikasi publik yang benerlah tiap buka aplikasi meminta verifikasi sms mulu lagi bikin kuis jari jariiiii lu meminta sms terus,wahaii pemuda bikin aplikasi publik yang benerlah tiap buka aplikasi meminta verifikasi sms mulu lagi bikin kuis jari jari lu meminta sms terus
211,seusai di update jadi leeeemoootttt,seusai di update jadi lemot


In [37]:
final_dataset = df[
    [
        "reviewId",
        "bank",
        "score",
        "year",
        "final_text"
    ]
].copy()

final_dataset.rename(
    columns={
        "final_text": "text"
    },
    inplace=True
)

final_dataset.head(30)

,reviewId,bank,score,year,text
0,46a9b775-12db-4cad-a259-83bdd6ff5b78,BCAMOBILE_REVIEWS,1,2025,bca enggak jelas sinyal bagus tapi merah mulu
1,fb449431-50de-41a5-9809-df47d670ee2a,BCAMOBILE_REVIEWS,1,2025,apa ini aplikasi bintang register mobile banking dari pagi sampai sore bilangnya tunggu indikator hijau padahal tidak ada indikator sama sekali sampai habis pulsa gue 20k
2,1390039e-cd94-4046-b864-523acc647bc9,BCAMOBILE_REVIEWS,1,2025,akun nya enggak bis akembali
3,5fa82eb0-e111-4a8f-b397-8a6fd37d1709,BCAMOBILE_REVIEWS,1,2025,makin parah setelah diupdate aplikasi nya ya allah
4,6881a68b-b85e-4ba0-9d8b-d5a2c21a6ca4,BCAMOBILE_REVIEWS,1,2025,indikator lampu merah terus mana sering begitu mulu lagi setiap mau dipakek kayak begitu mulu sampai berjam2 pada hak sinyal bagus apk sudah diperbarui kejadian terus berulang2 dan sering
5,d00b4bb9-41b6-438f-9a06-699a2aded5ff,BCAMOBILE_REVIEWS,2,2025,kenapa ya tiap mau buat sama bca selalu stuck di pembuatan sandi sama bca pas klik send tunggu lampu indikator hijau tapi letak indikator lampunya enggak ada sudah coba install uninstall restart hp tetap enggak mau jaringan wifi sama data juga oke enggak ada kendala ulang2 terus sampai berapa kali isi pulsa padahal dari dulu suka banget pakai nih aplikasi karena simpel ini ke reset karena ganti no hp tolong dong solusinya
6,7b37ce96-75c9-4854-8ee8-0a96299030a4,BCAMOBILE_REVIEWS,2,2025,indikator merah mulu setelah update
7,12880faf-c8a8-48ba-a35e-49d5ed40f677,BCAMOBILE_REVIEWS,1,2025,terlalu banyak error mengira ada update an lagi ternyata tidak
8,e62ba293-ed78-4fc3-b6b4-edb67d92bc02,BCAMOBILE_REVIEWS,3,2025,kenapa dari kemarin enggak bisa login apa sedang gangguan kah
9,a60449ad-d073-4537-af5c-77d70c859130,BCAMOBILE_REVIEWS,1,2025,sampah


In [38]:
final_dataset.shape

(59952, 5)

In [40]:
final_dataset.to_csv("preprocessed_data_2025.csv", index=False)